# Reverto — backtest & deal analysis

Exploratory notebook over the closed-deals ledger in `logs/reverto.db`. Covers:

1. Deal-history loading (with entry/exit-trigger JSON parse)
2. Win/loss + time-of-day statistics
3. Feature engineering via `ml.features.compute_features`
4. Correlation + feature-importance analysis
5. Market-regime clustering (`ml.market_regime`)
6. Parameter optimisation sketch with Optuna
7. XGBoost entry-filter training + cross-validation
8. Handover to the nightly pipeline

All heavy ML imports are wrapped in try/except so the notebook can be opened on a bare install — cells that need missing deps will print a clear install hint instead of crashing the kernel.

## 0. Setup

In [ ]:
# Uncomment to install the optional ML stack the first time you
# open this notebook on a fresh machine.
# !pip install optuna xgboost lightgbm scikit-learn \
#     pandas numpy matplotlib seaborn plotly joblib

In [ ]:
import sys
from pathlib import Path

# Make the Reverto package importable from the notebook dir.
sys.path.insert(0, str(Path.cwd().parent))

import sqlite3
import json
import numpy as np  # noqa: F401 — imported for downstream cells
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_style('darkgrid')
    HAS_PLOTS = True
except ImportError:
    HAS_PLOTS = False
    print('[info] matplotlib/seaborn not installed — plot cells will skip.')

In [ ]:
from pathlib import Path

# DB_PATH is a Path object (not a raw string) so fallback + resolve()
# work without having to re-construct the path later in the notebook.
DB_PATH = Path('../logs/reverto.db')
if not DB_PATH.exists():
    fallback = Path('..') / 'data' / 'reverto.db'
    if fallback.exists():
        DB_PATH = fallback

print(f'Using DB: {DB_PATH.resolve()}')
conn = sqlite3.connect(DB_PATH)

## 1. Deal history

In [ ]:
# Only closed deals have a realised PnL — they are the only rows
# useful for supervised training.
deals_df = pd.read_sql(
    '''
    SELECT id, bot_slug, bot_name, side, status, close_reason,
           opened_at, closed_at, initial_price, avg_entry,
           close_price, total_size, leverage, pnl_btc, pnl_pct,
           entry_trigger, exit_trigger
      FROM deals
     WHERE status = 'closed'
     ORDER BY opened_at
    ''',
    conn,
)

# opened_at / closed_at are stored as ISO strings — parse once.
deals_df['opened_dt'] = pd.to_datetime(deals_df['opened_at'], errors='coerce')
deals_df['closed_dt'] = pd.to_datetime(deals_df['closed_at'], errors='coerce')
deals_df['duration_hours'] = (
    (deals_df['closed_dt'] - deals_df['opened_dt']).dt.total_seconds() / 3600.0
)

# entry_trigger / exit_trigger are JSON blobs — unpack into flat columns.
def _parse_json(v):
    if not v:
        return {}
    try:
        return json.loads(v)
    except (TypeError, ValueError):
        return {}

deals_df['entry_group'] = deals_df['entry_trigger'].apply(
    lambda v: _parse_json(v).get('group_name', 'unknown') or 'unknown'
)
deals_df['exit_type'] = deals_df['exit_trigger'].apply(
    lambda v: _parse_json(v).get('type', 'unknown') or 'unknown'
)

print(f'Total deals: {len(deals_df)}')
if len(deals_df):
    first = deals_df['opened_dt'].min()
    last = deals_df['opened_dt'].max()
    print(f'Period:      {first} → {last}')
    print(f"Bots:        {sorted(deals_df['bot_slug'].unique())}")
deals_df.head()

## 2. Base statistics

In [ ]:
deals_df['won'] = deals_df['pnl_pct'] > 0

if len(deals_df):
    print('=== Deal statistics ===')
    print(f"Win rate:     {deals_df['won'].mean():.1%}")
    print(f"Avg PnL:      {deals_df['pnl_pct'].mean():.2f}%")
    print(f"Avg duration: {deals_df['duration_hours'].mean():.1f}h")
    print(f"Exit types:   {deals_df['exit_type'].value_counts().to_dict()}")

In [ ]:
if HAS_PLOTS and len(deals_df):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # PnL distribution.
    deals_df['pnl_pct'].hist(ax=axes[0, 0], bins=30, color='teal', alpha=0.75)
    axes[0, 0].set_title('PnL distribution (%)')
    axes[0, 0].axvline(0, color='black', linewidth=1)

    # Win rate by weekday (Mon..Sun).
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
                     'Friday', 'Saturday', 'Sunday']
    deals_df['weekday'] = deals_df['opened_dt'].dt.day_name()
    wr_day = deals_df.groupby('weekday')['won'].mean().reindex(weekday_order)
    wr_day.plot(kind='bar', ax=axes[0, 1], color='teal')
    axes[0, 1].set_title('Win rate by weekday')
    axes[0, 1].set_ylim(0, 1)

    # Win rate by hour.
    deals_df['hour'] = deals_df['opened_dt'].dt.hour
    wr_hour = deals_df.groupby('hour')['won'].mean()
    wr_hour.plot(ax=axes[1, 0], color='teal', marker='o')
    axes[1, 0].set_title('Win rate by hour of day (UTC)')
    axes[1, 0].set_ylim(0, 1)

    # Duration vs PnL scatter.
    axes[1, 1].scatter(deals_df['duration_hours'], deals_df['pnl_pct'],
                       alpha=0.5, color='teal')
    axes[1, 1].set_title('Duration (h) vs PnL (%)')
    axes[1, 1].axhline(0, color='black', linewidth=1)

    plt.tight_layout()
    plt.show()

## 3. Feature engineering

`ml.features.compute_features` computes a ~30-feature dict from a window of OHLCV candles. Wiring it up to real deals requires per-deal candle loading — left to the caller because the data source (ccxt / local CSV / Reverto `/api/candles`) depends on how you prefer to fetch history.

In [ ]:
from ml.features import compute_features, features_to_dataframe, MIN_CANDLES
from ml.candle_loader import load_candles_for_deal

print(f'compute_features requires >= {MIN_CANDLES} candles per deal.')

# Smoke test the loader on the first deal so a misconfigured
# exchange surfaces here rather than silently producing an empty
# feature frame downstream.
if len(deals_df) > 0:
    first_deal = deals_df.iloc[0].to_dict()
    test_candles = load_candles_for_deal(
        first_deal, symbol='BTC/USD', timeframe='1h', lookback_periods=100,
    )
    print(f"Loaded {len(test_candles)} candles for deal {first_deal.get('id')}")
    if test_candles:
        print(f'  first: {test_candles[0]}')
        print(f'  last:  {test_candles[-1]}')

feature_store = []
for _, deal in deals_df.iterrows():
    deal_dict = deal.to_dict()
    candles = load_candles_for_deal(
        deal_dict, symbol='BTC/USD', timeframe='1h', lookback_periods=100,
    )
    feats = compute_features(candles, deal_dict)
    if not feats:
        continue
    feats['pnl_pct'] = deal_dict.get('pnl_pct', 0.0)
    feats['won'] = bool(deal_dict.get('pnl_pct', 0.0) > 0)
    feature_store.append(feats)

features_df = features_to_dataframe(feature_store)
print(f'Built feature frame: {features_df.shape}')
if len(features_df):
    display(features_df.head())
else:
    print('no features — check candle_loader output + MIN_CANDLES threshold')

## 4. Correlation + feature importance

Only runs when `features_df` has rows. The classifier is a random forest — quick, robust, no tuning required. It gives a first-pass view of which features correlate with the win/loss label before you invest time in XGBoost + proper CV.

In [ ]:
if HAS_PLOTS and len(features_df) >= 20:
    corr = features_df.corr(numeric_only=True)
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, cmap='RdYlGn', center=0, vmin=-1, vmax=1,
                annot=False, cbar_kws={'shrink': 0.7})
    plt.title('Feature correlation matrix')
    plt.tight_layout()
    plt.show()
else:
    print('Correlation plot skipped — need >= 20 rows and matplotlib.')

In [ ]:
if len(features_df) >= 20:
    try:
        from sklearn.ensemble import RandomForestClassifier
        X = features_df.drop(['won', 'pnl_pct'], axis=1, errors='ignore')
        y = features_df['won'].astype(int)
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X, y)
        importance = pd.Series(rf.feature_importances_, index=X.columns)
        importance = importance.sort_values(ascending=False)
        if HAS_PLOTS:
            importance.head(20).plot(kind='barh', figsize=(10, 8),
                                     color='teal')
            plt.gca().invert_yaxis()
            plt.title('Top-20 feature importance (RandomForest)')
            plt.tight_layout()
            plt.show()
        else:
            print(importance.head(20))
    except ImportError:
        print('scikit-learn not installed — skipping feature importance.')
else:
    print('Need >= 20 labelled feature rows to fit a forest.')

## 5. Market regime detection

Uses `ml.market_regime` — KMeans on rolling volatility / trend / volume features. Labels are **positional** (cluster 0..3); reassign them to semantic names by inspecting the centroids.

In [ ]:
from ml import market_regime

# Plug in a big candle series here — the regime classifier needs
# at least a few hundred bars before rolling windows settle.
sample_candles = []  # e.g. load via ccxt.bitget().fetch_ohlcv(...)

if sample_candles:
    summary = market_regime.train_regime_model(sample_candles, n_regimes=4)
    print(summary)
    print('Latest regime:', market_regime.detect_current_regime(sample_candles))
else:
    print('Populate sample_candles to train a regime model.')

## 6. Parameter optimisation (Optuna)

Sketch — the objective currently returns 0 because hooking a real backtest runner is left to the operator. Replace the body with a call to `backtest.backtest_engine.BacktestEngine(...)` and return the Sortino (or any other) risk-adjusted metric.

In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            'rsi_threshold': trial.suggest_int('rsi_threshold', 20, 45),
            'tp_pct':        trial.suggest_float('tp_pct', 1.0, 5.0),
            'dca_spacing':   trial.suggest_float('dca_spacing', 0.5, 3.0),
            'dca_multiplier':trial.suggest_float('dca_multiplier', 1.0, 2.0),
            'max_dca_orders':trial.suggest_int('max_dca_orders', 2, 8),
        }
        # TODO: run Reverto backtest with these params and return
        # its Sortino ratio — for now this is a no-op stub so the
        # notebook can exercise the Optuna wiring.
        _ = params
        return 0.0

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=20, show_progress_bar=False)
    print('Best params:', study.best_params)
except ImportError:
    print('optuna not installed — run `pip install optuna` to enable.')

## 7. XGBoost entry filter

Trains an XGBoost classifier on the feature frame built above. Uses `TimeSeriesSplit` because deal order matters — a random K-fold would leak future trades into the training set and hugely overstate CV scores.

In [ ]:
try:
    import xgboost as xgb
    import joblib
    from sklearn.model_selection import TimeSeriesSplit, cross_val_score

    if len(features_df) >= 20:
        X = features_df.drop(['won', 'pnl_pct'], axis=1, errors='ignore')
        y = features_df['won'].astype(int)

        model = xgb.XGBClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, random_state=42,
            eval_metric='logloss', verbosity=0,
        )
        tscv = TimeSeriesSplit(n_splits=min(5, len(X) // 5))
        scores = cross_val_score(model, X, y, cv=tscv, scoring='roc_auc')
        print(f'ROC-AUC (time-series CV): {scores.mean():.3f} ± {scores.std():.3f}')

        model.fit(X, y)
        out_path = Path('..') / 'ml' / 'models' / 'entry_filter.pkl'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(model, out_path)
        print(f'Model saved to {out_path.resolve()}')
    else:
        print('Need >= 20 labelled rows — wire load_candles_for_deal().')
except ImportError:
    print('xgboost not installed — run `pip install xgboost` to enable.')

## 8. Nightly pipeline

For continuous training, run `ml/nightly_pipeline.py` under cron rather than this notebook. The pipeline re-executes steps 3 / 5 / 7 unattended and writes `ml/results_<bot_slug>.json` so an operator can diff yesterday's vs tonight's suggestions.

```
.venv/bin/python ml/nightly_pipeline.py --bot indi_group_test
```